In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [44]:
class IntrinsicValuationNetwork(nn.Module):

    def __init__(
        self,
        player_dim,
        num_archetypes,
        num_teams,
        embedding_dim=16,
    ):
        super().__init__()

        self.archetype_embedding = nn.Embedding(
            num_archetypes,
            embedding_dim
        )

        self.team_embedding = nn.Embedding(
            num_teams,
            embedding_dim
        )

        input_dim = player_dim + 2 * embedding_dim

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU()
        )

        self.mu_head = nn.Linear(64,1)
        self.sigma_head = nn.Linear(64,1)

    def forward(
        self,
        player_features,
        archetype,
        team
    ):

        a = self.archetype_embedding(archetype)
        t = self.team_embedding(team)

        x = torch.cat(
            [player_features,a,t],
            dim=1
        )

        h = self.network(x)

        mu = self.mu_head(h)

        sigma = F.softplus(
            self.sigma_head(h)
        ) + 1e-6

        return mu,sigma

In [45]:
class AuctionAdjustmentNetwork(nn.Module):

    def __init__(
        self,
        team_state_dim,
        auction_state_dim
    ):
        super().__init__()

        input_dim = (
            team_state_dim
            + auction_state_dim
        )

        self.network = nn.Sequential(

            nn.Linear(input_dim,128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,1)

        )

    def forward(
        self,
        team_state,
        auction_state
    ):

        x = torch.cat(
            [team_state,auction_state],
            dim=1
        )

        log_phi = self.network(x)

        return log_phi

In [46]:
class ValuationModel(nn.Module):

    def __init__(
        self,
        player_dim,
        team_state_dim,
        auction_state_dim,
        num_archetypes,
        num_teams,
        embedding_dim=16
    ):
        super().__init__()

        # Intrinsic valuation network
        self.intrinsic = IntrinsicValuationNetwork(
            player_dim=player_dim,
            num_archetypes=num_archetypes,
            num_teams=num_teams,
            embedding_dim=embedding_dim
        )

        # Auction adjustment network
        self.auction = AuctionAdjustmentNetwork(
            team_state_dim=team_state_dim,
            auction_state_dim=auction_state_dim
        )

    def forward(
        self,
        player_features,
        archetype,
        team,
        team_state,
        auction_state
    ):
    
        # Intrinsic valuation
        mu, sigma = self.intrinsic(
            player_features,
            archetype,
            team
        )
    
        # Auction adjustment
        log_phi = self.auction(
            team_state,
            auction_state
        )
    
        # Effective valuation
        mu_effective = mu + log_phi
    
        return {
            "mu": mu,
            "sigma": sigma,
            "log_phi": log_phi,
            "mu_effective": mu + log_phi
        }

In [47]:
player_features = torch.randn(200,120)

team_state = torch.randn(200,30)

auction_state = torch.randn(200,25)

team = torch.randint(0,10,(200,))

archetype = torch.randint(0,8,(200,))

In [48]:
model = ValuationModel(
    player_dim=120,
    team_state_dim=30,
    auction_state_dim=25,
    num_archetypes=8,
    num_teams=10,
    embedding_dim=16
)

results = model(
    player_features,
    archetype,
    team,
    team_state,
    auction_state
)

In [49]:
results.keys()

dict_keys(['mu', 'sigma', 'log_phi', 'mu_effective'])

In [50]:
import torch
from torch.utils.data import Dataset

class DummyAuctionDataset(Dataset):

    def __init__(
        self,
        n_samples=5000,
        player_dim=120,
        team_state_dim=30,
        auction_state_dim=25,
        num_archetypes=8,
        num_teams=10
    ):

        self.player_features = torch.randn(n_samples, player_dim)

        self.team_state = torch.randn(n_samples, team_state_dim)

        self.auction_state = torch.randn(n_samples, auction_state_dim)

        self.team = torch.randint(0, num_teams, (n_samples,))

        self.archetype = torch.randint(0, num_archetypes, (n_samples,))

        # Dummy interval data
        self.lower_bid = torch.rand(n_samples) * 100
        self.upper_bid = self.lower_bid + 0.5

        # Random winner indicator
        self.is_winner = torch.randint(0,2,(n_samples,)).bool()

    def __len__(self):
        return len(self.team)

    def __getitem__(self, idx):

        return {
            "player_features": self.player_features[idx],
            "team_state": self.team_state[idx],
            "auction_state": self.auction_state[idx],
            "team": self.team[idx],
            "archetype": self.archetype[idx],
            "lower_bid": self.lower_bid[idx],
            "upper_bid": self.upper_bid[idx],
            "winner": self.is_winner[idx]
        }

In [51]:
from torch.utils.data import DataLoader

dataset = DummyAuctionDataset()

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

In [52]:
import torch
import torch.nn as nn
from torch.distributions import LogNormal


class IntervalCensoredLoss(nn.Module):

    def __init__(self, eps=1e-10):
        super().__init__()
        self.eps = eps

    def forward(
        self,
        mu,
        sigma,
        lower_bid,
        upper_bid,
        winner
    ):

        # Remove trailing dimension if present
        mu = mu.squeeze(-1)
        sigma = sigma.squeeze(-1)

        dist = LogNormal(mu, sigma)

        loser_prob = (
            dist.cdf(upper_bid)
            -
            dist.cdf(lower_bid)
        )

        winner_prob = (
            1
            -
            dist.cdf(lower_bid)
        )

        likelihood = torch.where(
            winner,
            winner_prob,
            loser_prob
        )

        likelihood = torch.clamp(
            likelihood,
            min=self.eps
        )

        loss = -torch.log(
            likelihood
        ).mean()

        return {

            "loss": loss,

            "likelihood": likelihood.mean(),

            "winner_probability": winner_prob.mean(),

            "loser_probability": loser_prob.mean()

        }

In [53]:
# criterion = torch.nn.MSELoss()

# optimizer = torch.optim.Adam(
#     model.parameters(),
#     lr=1e-3
# )

# for epoch in range(20):

#     total_loss = 0

#     for batch in loader:

#         output = model(

#             batch["player_features"],

#             batch["archetype"],

#             batch["team"],

#             batch["team_state"],

#             batch["auction_state"]

#         )

#         prediction = output["mu_effective"].squeeze()

#         loss = criterion(
#             prediction,
#             batch["lower_bid"]
#         )

#         optimizer.zero_grad()

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     print(
#         epoch,
#         total_loss / len(loader)
#     )

In [54]:
criterion = IntervalCensoredLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [55]:
for epoch in range(20):

    total_loss = 0

    model.train()

    for batch in loader:

        output = model(

            batch["player_features"],

            batch["archetype"],

            batch["team"],

            batch["team_state"],

            batch["auction_state"]

        )

        stats = criterion(

            mu=output["mu_effective"],
        
            sigma=output["sigma"],
        
            lower_bid=batch["lower_bid"],
        
            upper_bid=batch["upper_bid"],
        
            winner=batch["winner"]
        
        )
        
        loss = stats["loss"]
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: {total_loss/len(loader):.4f}"
    )

Epoch 1: 4.3389
Epoch 2: 3.1279
Epoch 3: 3.0602
Epoch 4: 3.0304
Epoch 5: 2.9434
Epoch 6: 2.8488
Epoch 7: 2.7737
Epoch 8: 2.6424
Epoch 9: 2.6043
Epoch 10: 2.5287
Epoch 11: 2.4494
Epoch 12: 2.4208
Epoch 13: 2.4444
Epoch 14: 2.3669
Epoch 15: 2.2957
Epoch 16: 2.2954
Epoch 17: 2.2741
Epoch 18: 2.2506
Epoch 19: 2.2475
Epoch 20: 2.2619
